<a href="https://colab.research.google.com/github/luciacardozo472/TRABAJOP_IA/blob/main/sistema_multiagentes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 Sistema Multi-Agente: 3 Agentes ML

**Arquitectura basada en la presentación:**
- **Agente 1 - Normalizador:** limpia, imputa, escala, codifica
- **Agente 2 - Entrenador:** valida, entrena, selecciona modelo con embeddings/transformers
- **Agente 3 - Comunicador:** genera reporte en lenguaje natural

**Specs:** Transformers + Embeddings | Sin RAG | Dataset CSV ~100 celdas

## 📦 Instalación de dependencias

In [ ]:
!pip install -q transformers sentence-transformers scikit-learn pandas numpy torch

## 📊 Dataset CSV de ejemplo (~100 celdas, sin normalizar)

In [ ]:
import pandas as pd
import numpy as np
import io

# Dataset CSV de ejemplo sin normalizar (~100 celdas = ~20 filas x 5 columnas)
csv_data = """edad,salario,departamento,experiencia,satisfaccion
25,35000,IT,1,alta
32,,ventas,5,media
45,85000,IT,20,alta
28,42000,RRHH,3,
51,95000,Gerencia,25,alta
23,28000,ventas,,baja
38,67000,IT,12,alta
29,39000,RRHH,4,media
47,,ventas,18,media
34,54000,IT,9,alta
26,31000,ventas,2,baja
55,110000,Gerencia,30,alta
31,48000,IT,6,media
42,72000,RRHH,17,alta
27,,ventas,3,baja
36,58000,IT,10,media
49,88000,Gerencia,22,alta
33,51000,RRHH,8,media
24,29000,ventas,1,baja
40,75000,IT,15,alta
"""

df_raw = pd.read_csv(io.StringIO(csv_data))
print(f"Dataset cargado: {df_raw.shape[0]} filas x {df_raw.shape[1]} columnas")
print(f"Total celdas: {df_raw.size}")
print("\nPrimeras filas (sin normalizar):")
df_raw.head()

Dataset cargado: 20 filas x 5 columnas
Total celdas: 100

Primeras filas (sin normalizar):


,edad,salario,departamento,experiencia,satisfaccion
0,25,35000.0,IT,1.0,alta
1,32,NaN,ventas,5.0,media
2,45,85000.0,IT,20.0,alta
3,28,42000.0,RRHH,3.0,NaN
4,51,95000.0,Gerencia,25.0,alta


---
## 🤖 AGENTE 1 — Normalizador
> Limpia, imputa, escala y codifica el dataset

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer

class AgenteNormalizador:
    """Agente 1: Preprocesamiento completo del dataset."""

    def __init__(self):
        self.scaler = StandardScaler()
        self.label_encoders = {}
        self.log = []

    def limpiar(self, df):
        """Elimina duplicados y registra valores nulos."""
        nulos_antes = df.isnull().sum().sum()
        df = df.drop_duplicates()
        self.log.append(f"[Limpieza] Duplicados eliminados. Nulos encontrados: {nulos_antes}")
        return df

    def imputar(self, df):
        """Imputa valores faltantes: mediana para numéricos, moda para categóricos."""
        for col in df.columns:
            if df[col].isnull().any():
                if df[col].dtype in ['float64', 'int64']:
                    mediana = df[col].median()
                    df[col] = df[col].fillna(mediana)
                    self.log.append(f"[Imputación] '{col}': nulos → mediana ({mediana:.1f})")
                else:
                    moda = df[col].mode()[0]
                    df[col] = df[col].fillna(moda)
                    self.log.append(f"[Imputación] '{col}': nulos → moda ('{moda}')")
        return df

    def codificar(self, df, cols_categoricas):
        """Label encoding para columnas categóricas."""
        for col in cols_categoricas:
            le = LabelEncoder()
            df[col + '_enc'] = le.fit_transform(df[col].astype(str))
            self.label_encoders[col] = le
            self.log.append(f"[Codificación] '{col}' → '{col}_enc' | clases: {list(le.classes_)}")
        return df

    def escalar(self, df, cols_numericas):
        """StandardScaler sobre columnas numéricas."""
        df[cols_numericas] = self.scaler.fit_transform(df[cols_numericas])
        self.log.append(f"[Escalado] StandardScaler aplicado a: {cols_numericas}")
        return df

    def procesar(self, df):
        """Pipeline completo del Agente 1."""
        print("━" * 50)
        print("🤖 AGENTE 1 — Normalizador")
        print("━" * 50)

        df = self.limpiar(df.copy())
        df = self.imputar(df)

        cols_cat = ['departamento', 'satisfaccion']
        df = self.codificar(df, cols_cat)

        cols_num = ['edad', 'salario', 'experiencia']
        df = self.escalar(df, cols_num)

        for entry in self.log:
            print(" ", entry)

        print(f"\n✅ Dataset limpio: {df.shape[0]} filas x {df.shape[1]} columnas")
        return df


agente1 = AgenteNormalizador()
df_limpio = agente1.procesar(df_raw)
print("\nDataset listo para entrenamiento:")
df_limpio[['edad', 'salario', 'experiencia', 'departamento_enc', 'satisfaccion_enc']].head()

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🤖 AGENTE 1 — Normalizador
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  [Limpieza] Duplicados eliminados. Nulos encontrados: 5
  [Imputación] 'salario': nulos → mediana (54000.0)
  [Imputación] 'experiencia': nulos → mediana (9.0)
  [Imputación] 'satisfaccion': nulos → moda ('alta')
  [Codificación] 'departamento' → 'departamento_enc' | clases: ['Gerencia', 'IT', 'RRHH', 'ventas']
  [Codificación] 'satisfaccion' → 'satisfaccion_enc' | clases: ['alta', 'baja', 'media']
  [Escalado] StandardScaler aplicado a: ['edad', 'salario', 'experiencia']

✅ Dataset limpio: 20 filas x 7 columnas

Dataset listo para entrenamiento:


,edad,salario,experiencia,departamento_enc,satisfaccion_enc
0,-1.135122,-1.043876,-1.196083,1,0
1,-0.395973,-0.198092,-0.717650,3,2
2,0.976733,1.181872,1.076475,1,0
3,-0.818344,-0.732271,-0.956867,2,0
4,1.610289,1.627022,1.674517,0,0


---
## 🤖 AGENTE 2 — Entrenador
> Genera embeddings con Sentence Transformers, aplica validación cruzada, entrena y selecciona el mejor modelo

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
import numpy as np

class AgenteEntrenador:
    """Agente 2: Genera embeddings y entrena/selecciona el mejor modelo."""

    def __init__(self, modelo_embedding='all-MiniLM-L6-v2'):
        print("━" * 50)
        print("🤖 AGENTE 2 — Entrenador")
        print("━" * 50)
        print(f"  Cargando modelo de embeddings: {modelo_embedding}")
        self.embedder = SentenceTransformer(modelo_embedding)
        self.mejor_modelo = None
        self.mejor_score = 0
        self.metricas = {}

    def generar_embeddings(self, df, cols_originales):
        """Convierte filas en texto y genera embeddings semánticos (sin RAG)."""
        print("\n  [Embeddings] Generando representaciones vectoriales...")

        # Crear descripción textual de cada fila
        textos = df[cols_originales].apply(
            lambda row: f"empleado de {row['departamento']}, "
                        f"satisfaccion {row['satisfaccion']}, "
                        f"experiencia aproximada",
            axis=1
        ).tolist()

        embeddings = self.embedder.encode(textos, show_progress_bar=False)
        print(f"  [Embeddings] Shape: {embeddings.shape} (filas x dimensiones)")
        return embeddings

    def construir_features(self, df, embeddings):
        """Combina features numéricas + embeddings."""
        cols_num = ['edad', 'salario', 'experiencia',
                    'departamento_enc']
        X_num = df[cols_num].values
        X = np.hstack([X_num, embeddings])
        print(f"  [Features] Combinadas: {X_num.shape[1]} numéricas + {embeddings.shape[1]} embedding dims = {X.shape[1]} total")
        return X

    def entrenar_y_seleccionar(self, X, y):
        """Entrena múltiples modelos, valida con CV y selecciona el mejor."""
        print("\n  [Entrenamiento] Validación cruzada (cv=5) por modelo:")

        candidatos = {
            'Random Forest': RandomForestClassifier(n_estimators=50, random_state=42),
            'Gradient Boosting': GradientBoostingClassifier(n_estimators=50, random_state=42),
            'Logistic Regression': LogisticRegression(max_iter=500, random_state=42)
        }

        for nombre, modelo in candidatos.items():
            scores = cross_val_score(modelo, X, y, cv=5, scoring='accuracy')
            media = scores.mean()
            std = scores.std()
            self.metricas[nombre] = {'accuracy_media': round(media, 4), 'std': round(std, 4)}
            print(f"    {nombre}: accuracy={media:.4f} ± {std:.4f}")

            if media > self.mejor_score:
                self.mejor_score = media
                self.mejor_modelo_nombre = nombre
                self.mejor_modelo = modelo

        # Entrenar el mejor modelo en todo el dataset
        self.mejor_modelo.fit(X, y)
        print(f"\n  ✅ Mejor modelo: {self.mejor_modelo_nombre} (accuracy={self.mejor_score:.4f})")

    def procesar(self, df, df_original):
        """Pipeline completo del Agente 2."""
        cols_texto = ['departamento', 'satisfaccion']
        embeddings = self.generar_embeddings(df_original, cols_texto)

        X = self.construir_features(df, embeddings)
        y = df['satisfaccion_enc'].values

        self.entrenar_y_seleccionar(X, y)

        return {
            'modelo': self.mejor_modelo,
            'nombre_modelo': self.mejor_modelo_nombre,
            'metricas': self.metricas,
            'mejor_accuracy': self.mejor_score,
            'X': X,
            'y': y
        }


agente2 = AgenteEntrenador()
resultado = agente2.procesar(df_limpio, df_raw)

---
## 🤖 AGENTE 3 — Comunicador
> Genera un reporte automático en lenguaje natural usando un transformer de texto

In [ ]:
from transformers import pipeline

class AgenteComunicador:
    """Agente 3: Genera reporte en lenguaje natural."""

    def __init__(self):
        print("━" * 50)
        print("🤖 AGENTE 3 — Comunicador")
        print("━" * 50)
        print("  Cargando modelo de generación de texto...")
        # Usamos text-generation en lugar de text2text-generation
        self.generador = pipeline(
            "text-generation",
            model="sshleifer/tiny-gpt2",  # modelo ultraligero, ideal para Colab gratis
            max_new_tokens=80
        )

    def construir_contexto(self, resultado_agente2, df_original):
        metricas = resultado_agente2['metricas']
        mejor = resultado_agente2['nombre_modelo']
        acc = resultado_agente2['mejor_accuracy']

        ctx = (
            f"ML report: dataset with {df_original.shape[0]} employees. "
            f"Best model: {mejor}, accuracy: {acc:.4f}. "
            f"Embeddings used, no RAG. Summary:"
        )
        return ctx

    def generar_reporte(self, resultado_agente2, df_original):
        contexto = self.construir_contexto(resultado_agente2, df_original)

        print("  [Generando reporte...]")
        salida = self.generador(contexto, do_sample=False)[0]['generated_text']

        reporte_es = self._reporte_estructurado(resultado_agente2, df_original)

        print("\n" + "=" * 55)
        print("📄 REPORTE FINAL — Agente 3 Comunicador")
        print("=" * 55)
        print(reporte_es)
        print("\n[Texto generado por transformer]:")
        print(f"  {salida}")
        print("=" * 55)

        return reporte_es

    def _reporte_estructurado(self, resultado, df_original):
        metricas = resultado['metricas']
        mejor = resultado['nombre_modelo']
        acc = resultado['mejor_accuracy']

        lineas = [
            f"Dataset analizado: {df_original.shape[0]} registros, {df_original.shape[1]} variables.",
            f"Valores faltantes tratados: {df_original.isnull().sum().sum()} celdas.",
            f"",
            f"Modelos evaluados (CV 5-fold + embeddings semánticos):",
        ]
        for nombre, m in metricas.items():
            lineas.append(f"  • {nombre}: accuracy = {m['accuracy_media']} ± {m['std']}")

        lineas += [
            f"",
            f"Modelo seleccionado: {mejor}",
            f"Accuracy final: {acc:.4f} ({acc*100:.1f}%)",
            f"",
            f"Técnicas: SentenceTransformer embeddings, sin RAG.",
        ]
        return "\n".join(lineas)


agente3 = AgenteComunicador()
reporte = agente3.generar_reporte(resultado, df_raw)